# BudgiBrain
## Built with LangGraph, Semantic Memory, and Episodic Memory

### Dependencies and Requirements

In [3]:
# Insalling pip dependecies
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [4]:
# Importing all packages
from langgraph.graph import StateGraph, END
from langchain.docstore.document import Document
from langchain_core.messages import HumanMessage
from langchain_core.prompts import PromptTemplate
from langchain_core.tools import tool
from langchain_core.output_parsers import PydanticOutputParser
from langchain_groq import ChatGroq
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
from pydantic import BaseModel, Field
from typing import Optional, List
from datetime import datetime
from IPython.display import display, clear_output
from dotenv import load_dotenv
import ipywidgets as widgets
from uuid import uuid4
import os
import re

# Loading env variables
load_dotenv()

True

In [101]:
# Global variables
transaction_db = []

# Holds context when an 'add' intent is partially parsed and we need follow-up from user
pending_add_context = None

# Snapshot of the last final_state to recover context if needed across turns
last_state_snapshot = None

### Setting up Vector DB for Sematic Memory

In [6]:
# Some common tools that will be leveraged frequently

# Splitting of sample data into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=10)


# Setting up embedding model
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

/var/folders/60/jkpcvndn6j1bchnq17ptwvm80000gn/T/ipykernel_99735/182136857.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


In [102]:
# Sample training data (labelled user examples)
training_examples = [
    {
        "input": "Paid rent for August",
        "amount": None,
        "item_name": "rent",
        "category": "Rent"
    },
    {
        "input": "Gave landlord 12,000 for rent",
        "amount": 12000,
        "item_name": "rent",
        "category": "Rent"
    },
    {
        "input": "Netflix charged my card",
        "amount": None,
        "item_name": "Netflix",
        "category": "Subscriptions"
    },
    {
        "input": "Monthly subscription to Netflix",
        "amount": None,
        "item_name": "Netflix",
        "category": "Subscriptions"
    },
    {
        "input": "Bought veggies and snacks from DMart",
        "amount": None,
        "item_name": "DMart",
        "category": "Groceries"
    },
    {
        "input": "Shopping at Walmart – groceries and drinks",
        "amount": None,
        "item_name": "Walmart",
        "category": "Groceries"
    },
    {
        "input": "Recharge for electricity board",
        "amount": None,
        "item_name": "electricity board",
        "category": "Utilities"
    },
    {
        "input": "TNEB current bill paid",
        "amount": None,
        "item_name": "TNEB",
        "category": "Utilities"
    },
    {
        "input": "Watched Barbie movie at INOX",
        "amount": None,
        "item_name": "INOX",
        "category": "Shopping & Entertainment"
    },
    {
        "input": "Cinema with friends at PVR",
        "amount": None,
        "item_name": "PVR",
        "category": "Shopping & Entertainment"
    },
    {
        "input": "Renewed Tata AIG insurance for car",
        "amount": None,
        "item_name": "Tata AIG",
        "category": "Insurance"
    },
    {
        "input": "Paid ICICI car insurance",
        "amount": None,
        "item_name": "ICICI",
        "category": "Insurance"
    },
    {
        "input": "Consultation at Apollo Hospital",
        "amount": None,
        "item_name": "Apollo Hospital",
        "category": "Health"
    },
    {
        "input": "Doctor visit charges",
        "amount": None,
        "item_name": "Doctor",
        "category": "Health"
    },
    {
        "input": "Recharged metro card",
        "amount": None,
        "item_name": "metro card",
        "category": "Transport"
    },
    {
        "input": "Added ₹200 to metro pass",
        "amount": 200,
        "item_name": "metro pass",
        "category": "Transport"
    },
]

# Convert sample data to documents with category label as metadata
docs = []
for example in training_examples:
    docs.append(Document(
        page_content=example["input"],
        metadata={
            "doc_id": str(uuid4()),
            "input": example["input"],
            "amount": example["amount"],
            "item_name": example["item_name"],
            "category": example["category"],
            "action": "add",
            "source": "training_data"
        }
    ))

split_docs = text_splitter.split_documents(docs)

In [103]:
# Load or create FAISS vector store
if os.path.exists("faiss_store/index.faiss"):
    db = FAISS.load_local(
        "faiss_store",
        embeddings=embedding_model,
        allow_dangerous_deserialization=True
    )
    print("Using existing FAISS vector DB")
else:
    db = FAISS.from_documents(docs, embedding_model)
    db.save_local("faiss_store")
    print("Created and saved new FAISS vector DB from training examples")

Created and saved new FAISS vector DB from training examples


### Setting up LLM

In [104]:
# Initialize LLM
LLM = ChatGroq(
    model_name=os.environ.get("LITELLM_MODEL"),
    groq_api_key=os.environ.get("GROQ_API_KEY")
)

### API Tools used by the LLM

In [105]:
# API Tools that will be used by the LLM
@tool
def add_transaction(amount: Optional[float], category: Optional[str], item_name: Optional[str], input: str) -> dict:
    """Add a new transaction to the transaction_db. Amount, category and item_name may be omitted."""
    
    # Create a persistent id for the vector store and transaction
    doc_id = str(uuid4())

    transaction = {
        "datetime": datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        "amount": amount,
        "category": category,
        "item_name": item_name,
        "input": input,
        "doc_id": doc_id,
    }
    
    # Save to in-memory DB
    transaction_db.append(transaction)

    # Save to vector DB
    doc = Document(
        page_content=input,
        metadata={
            "doc_id": doc_id,
            "amount": amount,
            "category": category,
            "item_name": item_name,
            "datetime": transaction["datetime"],
            "source": "user"
        }
    )
    db.add_documents([doc], ids=[doc_id])
    db.save_local("faiss_store")

    print("✅ Stored transaction in FAISS vector DB")
    return transaction

@tool
def edit_transaction(
    input: str,
    amount: Optional[float] = None,
    category: Optional[str] = None,
    item_name: Optional[str] = None
) -> dict:
    """
    Edit a transaction by searching for the most similar past transaction using the user input.
    Can update amount, category, and/or item_name if provided.
    """
    # Retrieve similar transactions from vector DB
    similar_docs = db.similarity_search(input, k=1)
    if not similar_docs:
        return {"error": "No matching transaction found to edit."}
    
    best_match = similar_docs[0]
    metadata = best_match.metadata
    original_input = best_match.page_content

    # Locate and update the transaction in memory
    for i, t in enumerate(transaction_db):
        # Prefer matching by doc_id when available; fall back to input+datetime
        same_doc = (t.get("doc_id") and t.get("doc_id") == metadata.get("doc_id"))
        same_input_dt = (
            t.get("input") == original_input and t.get("datetime") == metadata.get("datetime")
        )
        if same_doc or same_input_dt:
            updated_transaction = t.copy()
            
            # Update fields if new values are provided
            if amount is not None:
                updated_transaction["amount"] = amount
            if category is not None:
                updated_transaction["category"] = category
            if item_name is not None:
                updated_transaction["item_name"] = item_name

            # Update both in-memory DB and vector DB
            transaction_db[i] = updated_transaction
            
            # Delete old doc from FAISS using doc_id (if available)
            doc_id = metadata.get("doc_id")
            if doc_id:
                try:
                    db.delete([doc_id])
                except ValueError:
                    # If id not found, fall back to similarity removal of original_input
                    try:
                        # Find a close match and delete its id if present
                        candidates = db.similarity_search(original_input, k=3)
                        for c in candidates:
                            cid = c.metadata.get("doc_id")
                            if cid:
                                try:
                                    db.delete([cid])
                                    break
                                except Exception:
                                    continue
                    except Exception:
                        pass
            else:
                print("⚠️ Warning: doc_id missing; skipping delete")

            # Add updated doc with same doc_id so we replace the vector
            new_doc_id = metadata.get("doc_id") or str(uuid4())
            new_doc = Document(
                page_content=input,
                metadata={
                    "doc_id": new_doc_id,
                    "datetime": updated_transaction["datetime"],
                    "amount": updated_transaction["amount"],
                    "category": updated_transaction["category"],
                    "item_name": updated_transaction["item_name"],
                    "input": input,
                    "source": "user"
                }
            )
            db.add_documents([new_doc], ids=[new_doc_id])
            db.save_local("faiss_store")  # Persist changes
            
            return {
                "updated_transaction": updated_transaction,
                "matched_on": original_input
            }

    return {"error": "Transaction found in vector DB but not in in-memory DB."}

@tool
def search_transaction_by_category(category: str) -> List[dict]:
    """Search for transactions by category."""
    results = []
    for t in transaction_db:
        if (t['category'] == category):
            results.append(t)
    return results


@tool
def get_recent_similar_transactions(input: str, k: int = 3) -> List[dict]:
    """
    Get recent transactions similar to the user input from the vector DB.
    Returns the top-k most recent similar transactions.
    """
    similar_docs = db.similarity_search(input, k=10, filter={"source": "user"})  # get more for filtering

    if not similar_docs:
        return []

    # Filter and sort by datetime (descending)
    parsed_docs = []
    for doc in similar_docs:
        metadata = doc.metadata
        dt_str = metadata.get("datetime")
        try:
            dt = datetime.strptime(dt_str, '%Y-%m-%d %H:%M:%S')
        except Exception:
            dt = datetime.min  # fallback for malformed datetimes

        parsed_docs.append({
            "input": doc.page_content,
            "amount": metadata.get("amount"),
            "category": metadata.get("category"),
            "item_name": metadata.get("item_name"),
            "datetime": dt_str,
            "datetime_obj": dt
        })

    # Sort by most recent
    parsed_docs.sort(key=lambda x: x["datetime_obj"], reverse=True)

    # Return top-k without the datetime_obj helper
    return [{k: v for k, v in doc.items() if k != "datetime_obj"} for doc in parsed_docs[:k]]


tools = [add_transaction, edit_transaction, search_transaction_by_category, get_recent_similar_transactions]

### Building actual Graph

In [106]:
# State Object that is passed in Graph
class TransactionState(BaseModel):
    input: str = ""
    amount: Optional[float] = None
    category: Optional[str] = None
    item_name: Optional[str] = None
    action: Optional[str] = None
    result: Optional[dict] = None
    parse_attempts: int = 0
    needs_followup: bool = False
    followup_prompt: Optional[str] = None

# Output Parser 
class TransactionParse(BaseModel):
    action: str
    amount: Optional[float]
    category: Optional[str]
    item_name: Optional[str]

def retrieve_similar_examples(query: str, k: int = 3) -> List[Document]:
    return db.similarity_search(query, k=k)

# Node function of Graph
def llm_parse_node(state: TransactionState) -> TransactionState:
    print("➡️ Running llm_parse_node")
    state.parse_attempts += 1

    # Fetch similar examples
    similar_docs = retrieve_similar_examples(state.input)
    context = "\n".join(
        f"- {doc.page_content} ({doc.metadata.get('category')})"
        for doc in similar_docs
    )

    print("🧠 Similar examples:\n", context)

    # Compose context-enhanced prompt
    prompt_template = PromptTemplate.from_template(
        """
        You are a transaction assistant. Use the examples below to help understand the user's input.

        Examples:
        {context}

        Now extract the following fields from the user input:
        - action: one of "add", "edit", "search_by_category", "get_recent"
        - amount: the transaction amount (float or null)
        - category: the transaction category mentioned below
        - item_name: the item name (string or null)

        Categories:
        - Rent
        - Insurance
        - Utilities
        - Shopping & Entertainment
        - Groceries
        - Subscriptions
        - Transport
        - Health

        If a field is missing, set it to null.

        User input: {input}

        {format_instructions}
        """
    )

    parser = PydanticOutputParser(pydantic_object=TransactionParse)
    format_instructions = parser.get_format_instructions()
    chain = prompt_template | LLM | parser
    parsed: TransactionParse = chain.invoke({"input": state.input, "context": context, "format_instructions": format_instructions})
    

    print("🧠 Parsed Output:", parsed)

    state.action = parsed.action
    state.amount = parsed.amount
    state.category = parsed.category
    state.item_name = parsed.item_name

    return state

def add_node(state: TransactionState) -> TransactionState:
    result = add_transaction.invoke({
				"amount": state.amount,
				"category": state.category,
				"item_name": state.item_name,
				"input": state.input
		})
    state.result = result
    return state


def edit_node(state: TransactionState) -> TransactionState:
    result = edit_transaction.invoke({
        "input": state.input,
				"category": state.category,
				"item_name": state.item_name,
				"amount": state.amount
		})
    state.result = result
    return state

def search_node_by_category(state: TransactionState) -> TransactionState:
    result = search_transaction_by_category.invoke({
				"category": state.category
		})
    state.result = result
    return state

def get_recent_node(state: TransactionState) -> TransactionState:
    result = get_recent_similar_transactions.invoke({
        "input": state.input,
        "k": 1  
    })
    state.result = result
    return state

# Decision Function
def decide_next(state: TransactionState):
    print("🔄 Deciding next step...")
    print("State:", state)

    max_attempts = 1

    # 1) Add flow: if missing required fields after attempts, ask user for follow-up
    if state.action == "add" and (state.amount is None or state.item_name is None):
        if state.parse_attempts >= max_attempts:
            missing = []
            if state.amount is None:
                missing.append("amount")
            if state.item_name is None:
                missing.append("item name")
            missing_str = " and ".join(missing)
            state.needs_followup = True
            state.followup_prompt = (
                f"I can add this under {state.category or 'unspecified category'}. "
                f"Please provide the {missing_str}. If you want to proceed without it, reply 'skip'."
            )
            print("🟡 Asking user for follow-up:", state.followup_prompt)
            return END

    # 2) Re-enter parse if still missing fields for add/edit
    if state.action == "add" and (state.amount is None or state.category is None or state.item_name is None):
        print("🔁 Re-entering llm_parse (missing add fields)")
        return "llm_parse"
    if state.action == "edit" and (state.category is None or state.item_name is None):
        print("🔁 Re-entering llm_parse (missing edit fields)")
        return "llm_parse"

    # 3) Route to concrete actions as soon as recognized
    if state.action == "add":
        print("✅ Going to add")
        return "add"
    if state.action == "edit":
        print("✅ Going to edit")
        return "edit"
    if state.action == "search_by_category":
        print("✅ Going to search by category")
        return "search_by_category"
    if state.action == "get_recent":
        print("✅ Going to get_recent")
        return "get_recent"

    # 4) For unknown intents, only then apply the generic attempts guard
    if state.parse_attempts >= max_attempts:
        state.result = {
            "error": "❌ Unable to understand input after multiple attempts. Please rephrase."
        }
        return END

    print("⏹ Ending graph")
    return END

# Build the Graph
def build_transaction_graph():
    graph = StateGraph(TransactionState)
    graph.add_node("llm_parse", llm_parse_node)
    graph.add_node("add", add_node)
    graph.add_node("edit", edit_node)
    graph.add_node("search_by_category", search_node_by_category)
    graph.add_node("get_recent", get_recent_node)
    graph.add_edge("get_recent", END)
    graph.add_edge("add", END)
    graph.add_edge("edit", END)
    graph.add_edge("search_by_category", END)
    graph.add_conditional_edges("llm_parse", decide_next)
    graph.set_entry_point("llm_parse")
    return graph.compile()

transaction_graph = build_transaction_graph()

# Example to call

def format_natural_language_response(final_state, result) -> str:
    action = final_state.get("action")
    category = final_state.get("category")
    item_name = final_state.get("item_name")
    amount = final_state.get("amount")

    # If a follow-up is needed, surface the prompt
    if final_state.get("needs_followup") and final_state.get("followup_prompt"):
        return final_state.get("followup_prompt")

    if isinstance(result, dict) and "error" in result:
        return f"{result['error']}"

    if action == "add" and isinstance(result, dict):
        amt_text = f"{amount}" if amount is not None else "an unspecified amount"
        item_text = f" for {item_name}" if item_name else ""
        cat_text = f" in category {category}" if category else ""
        return f"Added a transaction of {amt_text}{cat_text}{item_text}."

    if action == "edit" and isinstance(result, dict):
        updated = result.get("updated_transaction", {})
        amt_text = f"{updated.get('amount')}" if updated.get("amount") is not None else "an unspecified amount"
        item_text = f" for {updated.get('item_name')}" if updated.get("item_name") else ""
        cat_text = f" in category {updated.get('category')}" if updated.get("category") else ""
        return f"Updated the transaction to {amt_text}{cat_text}{item_text}."

    if action == "search_by_category" and isinstance(result, list):
        n = len(result)
        if n == 0:
            return f"No transactions found in category {category}."
        lines = [f"Found {n} transaction{'s' if n != 1 else ''} in category {category}:"]
        for t in result[:5]:
            amt = t.get("amount")
            name = t.get("item_name") or "unspecified item"
            dt = t.get("datetime")
            amt_text = f"{amt}" if amt is not None else "unspecified amount"
            lines.append(f"- {dt}: {name} ({amt_text})")
        if n > 5:
            lines.append(f"...and {n-5} more.")
        return "\n".join(lines)

    if action == "get_recent" and isinstance(result, list):
        if not result:
            return "No similar recent transactions found."
        lines = ["Most recent similar transaction(s):"]
        for t in result[:3]:
            amt = t.get("amount")
            name = t.get("item_name") or "unspecified item"
            cat = t.get("category")
            dt = t.get("datetime")
            amt_text = f"{amt}" if amt is not None else "unspecified amount"
            cat_text = f" in {cat}" if cat else ""
            lines.append(f"- {dt}: {name}{cat_text} ({amt_text})")
        return "\n".join(lines)

    # Fallbacks
    if isinstance(result, dict):
        parts = []
        for k, v in result.items():
            parts.append(f"{k}: {v}")
        return "Result:\n" + "\n".join(parts)

    return str(result)


def call_transaction_agent(user_input: str):
    global pending_add_context
    print(f"🔍 User Input: {user_input}\n")

    # If we are waiting for follow-up (amount/item) and the user responded
    if pending_add_context:
        follow_raw = user_input.strip()
        follow = follow_raw.lower()
        ctx = pending_add_context

        # If the reply indicates skipping missing fields
        if follow in {"skip", "no", "n", "none", "na"}:
            pending_add_context = None
            result = add_transaction.invoke({
                "amount": ctx.get("amount"),
                "category": ctx.get("category"),
                "item_name": ctx.get("item_name"),
                "input": ctx.get("input")
            })
            final_state = {
                "action": "add",
                "amount": ctx.get("amount"),
                "category": ctx.get("category"),
                "item_name": ctx.get("item_name"),
                "result": result
            }
            message = format_natural_language_response(final_state, result)
            print(message)
            return message, result

        # Try to parse the reply to fill in missing fields
        if ctx.get("amount") is None:
            # Extract numeric value if present
            digits = re.sub(r"[^0-9.]+", "", follow)
            if digits:
                try:
                    ctx["amount"] = float(digits)
                except Exception:
                    pass

        if ctx.get("item_name") is None and ctx.get("amount") is not None and not follow in {"skip", "no", "n", "none", "na"}:
            # If amount is now present and item is still missing, and user typed text, treat as item_name
            if not re.fullmatch(r"[0-9.]+", follow):
                ctx["item_name"] = follow_raw

        # If we still miss item_name and user typed non-numeric text first
        if ctx.get("item_name") is None and not re.fullmatch(r"[0-9.]+", follow) and follow not in {"skip", "no", "n", "none", "na"}:
            ctx["item_name"] = follow_raw

        # Proceed when we have at least category and one of amount/item_name
        if ctx.get("category") and (ctx.get("amount") is not None or ctx.get("item_name") is not None):
            pending_add_context = None
            result = add_transaction.invoke({
                "amount": ctx.get("amount"),
                "category": ctx.get("category"),
                "item_name": ctx.get("item_name"),
                "input": ctx.get("input")
            })
            final_state = {
                "action": "add",
                "amount": ctx.get("amount"),
                "category": ctx.get("category"),
                "item_name": ctx.get("item_name"),
                "result": result
            }
            message = format_natural_language_response(final_state, result)
            print(message)
            return message, result

        # Still missing; ask again
        missing = []
        if ctx.get("amount") is None:
            missing.append("amount")
        if ctx.get("item_name") is None:
            missing.append("item name")
        prompt = f"Please provide the {' and '.join(missing)}. If you want to proceed without it, reply 'skip'."
        print(prompt)
        return prompt, None

    # Normal flow
    state = TransactionState(input=user_input)
    final_state = transaction_graph.invoke(state)

    # Cache last_state for robustness
    globals()["last_state_snapshot"] = final_state

    # If follow-up needed, save context and prompt user (primary path)
    if final_state.get("needs_followup"):
        pending_add_context = {
            "input": user_input,
            "amount": final_state.get("amount"),
            "category": final_state.get("category"),
            "item_name": final_state.get("item_name")
        }
        message = format_natural_language_response(final_state, final_state.get("result"))
        print(message)
        return message, None

    # Fallback: if graph ended without result and we can infer missing add fields, prompt and cache context
    if (
        final_state.get("result") is None
        and final_state.get("action") == "add"
        and (final_state.get("amount") is None or final_state.get("item_name") is None)
    ):
        pending_add_context = {
            "input": user_input,
            "amount": final_state.get("amount"),
            "category": final_state.get("category"),
            "item_name": final_state.get("item_name")
        }
        missing = []
        if final_state.get("amount") is None:
            missing.append("amount")
        if final_state.get("item_name") is None:
            missing.append("item name")
        missing_str = " and ".join(missing)
        prompt = (
            f"I can add this under {final_state.get('category') or 'unspecified category'}. "
            f"Please provide the {missing_str}. If you want to proceed without it, reply 'skip'."
        )
        print(prompt)
        return prompt, None

    result = final_state.get("result")
    message = format_natural_language_response(final_state, result)
    print(message)

    return message, result

In [107]:
# Interactive_chat
def interactive_chat():
    input_box = widgets.Text(
        description='Prompt:',
        placeholder='e.g. Add 500 for groceries as milk',
        layout=widgets.Layout(width='90%')
    )
    output_area = widgets.Output()

    def on_enter(_):
        user_input = input_box.value.strip()
        if user_input:
            message, _ = call_transaction_agent(user_input)
            with output_area:
                clear_output(wait=True)
                print(message)
            input_box.value = ''

    input_box.on_submit(on_enter)
    display(input_box, output_area)
    
    print("Tip: When asked for missing amount or item, you can type a number (e.g., 500) or 'skip'.")

In [113]:
call_transaction_agent("edit category of car transaction to transport")

🔍 User Input: edit category of car transaction to transport

➡️ Running llm_parse_node
🧠 Similar examples:
 - add a transaction of buying car for 1000 (Shopping & Entertainment)
- add a transaction of bus for 1000 (Transport)
- add transaction of jeans (Shopping & Entertainment)


/opt/homebrew/Caskroom/miniconda/base/envs/budgibot-backend/lib/python3.11/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


🧠 Parsed Output: action='edit' amount=None category='Transport' item_name='car'
🔄 Deciding next step...
State: input='edit category of car transaction to transport' amount=None category='Transport' item_name='car' action='edit' result=None parse_attempts=1 needs_followup=False followup_prompt=None
✅ Going to edit
Updated the transaction to 1000.0 in category Transport for car.


/opt/homebrew/Caskroom/miniconda/base/envs/budgibot-backend/lib/python3.11/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


('Updated the transaction to 1000.0 in category Transport for car.',
 {'updated_transaction': {'datetime': '2025-08-11 13:22:37',
   'amount': 1000.0,
   'category': 'Transport',
   'item_name': 'car',
   'input': 'add a transaction of buying car for 1000',
   'doc_id': '2b86aa2b-cee3-4365-8b8a-0b745a82dacf'},
  'matched_on': 'add a transaction of buying car for 1000'})

In [56]:
interactive_chat()

/var/folders/60/jkpcvndn6j1bchnq17ptwvm80000gn/T/ipykernel_99735/3995379316.py:19: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  input_box.on_submit(on_enter)


Text(value='', description='Prompt:', layout=Layout(width='90%'), placeholder='e.g. Add 500 for groceries as m…

Output()

Tip: When asked for missing amount or item, you can type a number (e.g., 500) or 'skip'.


### Utility Blocks

In [ ]:
# Print size of locally stored vector db
db = FAISS.load_local(
      "faiss_store", 
      embeddings=embedding_model,
      allow_dangerous_deserialization=True  # explicitly allows loading .pkl safely
    )

# Print number of vectors
print(f"Number of vectors in FAISS DB: {len(db.index_to_docstore_id)}")

# Function to get size of folder
def get_folder_size(path):
    return sum(
        os.path.getsize(os.path.join(dirpath, filename))
        for dirpath, _, filenames in os.walk(path)
        for filename in filenames
    )

# Calculate and print DB size
size_bytes = get_folder_size("faiss_store")
size_mb = size_bytes / (1024 * 1024)
print(f"FAISS DB size on disk: {size_mb:.2f} MB")

In [114]:
# View transaction_db
for transaction in transaction_db:
    print("Date/Time:", transaction['datetime'])
    print("Category:", transaction['category'])
    print("Input:", transaction['input'])
    print("Amount:", transaction['amount'])
    print("-" * 30)  # Separator for readability

Date/Time: 2025-08-11 13:21:13
Category: Shopping & Entertainment
Input: add transaction of jeans
Amount: 100.0
------------------------------
Date/Time: 2025-08-11 13:22:37
Category: Transport
Input: add a transaction of buying car for 1000
Amount: 1000.0
------------------------------
Date/Time: 2025-08-11 13:23:11
Category: Transport
Input: add a transaction of bus for 1000
Amount: 1000.0
------------------------------


In [ ]:
# # Delete locally stored vector db
# import shutil

# shutil.rmtree("faiss_store")
# print("Local FAISS vector DB deleted.")

Local FAISS vector DB deleted.
